# Plate 01 · Dataset Preparation (Kuwaiti plates)

Builds one **detection + corner** dataset (YOLO-pose format) and one **cell recognition** dataset.

A Kuwaiti plate has a **code** (1-2 digits) and a **number** (up to 5 digits), each printed twice:
Arabic-Indic digits on top, English digits below. No letters. See `Common_Utils/Kuwait_Plate_Layout.py`.

| Source | Use | Licence note |
|---|---|---|
| **Pilot-station capture** (Kuwait, with the owner's written consent): 3-5 k frames from the bay cameras/phones | detector + recogniser + evaluation | ours, consented |
| Public Kuwait / GCC plate sets (search Kaggle and Roboflow Universe for "Kuwait license plate") | detector + recogniser | **check each licence** before use |
| **Open Images v7** class *Vehicle registration plate* | detector pre-training (any country) | CC-BY 4.0 |
| Synthetic Kuwaiti plates (notebook 02) | recogniser, rare plate types, night/rain/glare | ours |
| Station corrections in production (`ai_training_samples`) | per-station fine-tune (notebook 07) | the station's own data |

Split by **plate key** (`stable_split`) so the same car never appears in train and test.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import json, shutil
import pandas as pd
from Common_Utils.Data_Utils import list_images, stable_split

RAW = DATA / "Plates" / "Raw"
DET = DATA / "Plates" / "Detection"      # YOLO format
REC = DATA / "Plates" / "Recognition"    # crops + labels.csv
for d in (RAW, DET, REC):
    d.mkdir(parents=True, exist_ok=True)

### 1. Download (fill in the dataset slugs you have accepted)

In [ ]:
KAGGLE_SLUGS = [
    # "owner/kuwait-license-plates",   # <- exact slugs you have checked the licence of
]
import subprocess
for slug in KAGGLE_SLUGS:
    target = RAW / slug.split("/")[-1]
    if not target.exists():
        subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", str(target), "--unzip"], check=True)
# Pilot-station frames go to RAW / "Pilot_<station>" (copied from the capture app), labelled in CVAT / Label Studio.
print(sorted(p.name for p in RAW.iterdir()))

### 2. Convert annotations to YOLO-pose
Label each plate with its **box + 4 corners** (top-left, top-right, bottom-right, bottom-left) and its text
`CODE-NUMBER` (e.g. `12-34567`). The corners let the pipeline straighten the plate to the canonical layout.

Line format: `class xc yc w h  x1 y1 v1  x2 y2 v2  x3 y3 v3  x4 y4 v4` (normalised, v = 2 visible / 1 occluded).
Classes (plate type): `0 private, 1 taxi, 2 commercial_transport, 3 government, 4 other` -
**confirm the current Kuwaiti plate types / colours against MOI rules in Phase 0.**

In [ ]:
from Common_Utils.Plate_Normalizer import normalize

PLATE_CLASSES = ["private", "taxi", "commercial_transport", "government", "other"]

def adapter_example(source_dir):
    """Template adapter - replace with the real export format (CVAT / Label Studio json, VOC xml ...).
    Must yield (image_path, [(class_id, xc, yc, w, h, [x1,y1,v1, ..., x4,y4,v4]), ...], plate_key)."""
    for label_file in sorted(Path(source_dir).rglob("*.txt")):
        image = label_file.with_suffix(".jpg")
        if not image.exists():
            continue
        objects = []
        for line in label_file.read_text().splitlines():
            v = line.split()
            if len(v) >= 17:
                objects.append((int(v[0]), *map(float, v[1:5]), list(map(float, v[5:17]))))
        yield image, objects, normalize(label_file.stem)   # file stem holds the plate text in this template

def write_yolo(records):
    rows = []
    for image, objects, key in records:
        if not key:
            continue                                  # unreadable / invalid label -> skip
        split = stable_split(key)
        (DET / "images" / split).mkdir(parents=True, exist_ok=True)
        (DET / "labels" / split).mkdir(parents=True, exist_ok=True)
        dst = DET / "images" / split / image.name
        shutil.copy2(image, dst)
        (DET / "labels" / split / f"{image.stem}.txt").write_text("\n".join(
            f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f} " + " ".join(f"{k:.6f}" for k in kp) for c, x, y, w, h, kp in objects))
        rows.append({"image": dst.name, "split": split, "plates": len(objects), "key": key})
    return pd.DataFrame(rows)

frames = [write_yolo(adapter_example(src)) for src in RAW.iterdir() if src.is_dir()]
index = pd.concat(frames) if frames else pd.DataFrame(columns=["image", "split", "plates", "key"])
index.to_csv(DET / "index.csv", index=False)
index.groupby("split").size() if len(index) else "no data yet"

In [ ]:
(DET / "plates.yaml").write_text(f"""path: {DET.as_posix()}
train: images/train
val: images/val
test: images/test
kpt_shape: [4, 3]          # 4 corners (x, y, visibility)
flip_idx: [1, 0, 3, 2]     # horizontal flip swaps left/right corners
names: {json.dumps(dict(enumerate(PLATE_CLASSES)))}
""")
print((DET / "plates.yaml").read_text())

### 3. Cell recognition crops
For every labelled plate: warp it with its 4 corners to the canonical layout, cut the **4 cells**
(code / number × Arabic / English) and save each with its own text. The recogniser learns short digit strips only.
Arabic cells are labelled with Arabic-Indic digits, English cells with English digits.

In [ ]:
import cv2
from Common_Utils.Kuwait_Plate_Layout import warp, split_cells, cell_input
from Common_Utils.Plate_Normalizer import parse, to_arabic_digits

def crop_cells(det_root=DET, out=REC):
    rows = []
    index = pd.read_csv(det_root / "index.csv")
    for _, r in index.iterrows():
        label = det_root / "labels" / r["split"] / f"{Path(r['image']).stem}.txt"
        image = cv2.imread(str(det_root / "images" / r["split"] / r["image"]))
        parsed = parse(r["key"])
        if image is None or not label.exists() or not parsed:
            continue
        code, number = parsed
        texts = {"code_ar": to_arabic_digits(code), "code_lat": code, "num_ar": to_arabic_digits(number), "num_lat": number}
        h, w = image.shape[:2]
        for i, line in enumerate(label.read_text().splitlines()):
            v = list(map(float, line.split()))
            corners = [(v[5 + 3 * k] * w, v[6 + 3 * k] * h) for k in range(4)]
            for cell_name, crop in split_cells(warp(image, corners)).items():
                crop = cell_input(crop)
                name = f"{Path(r['image']).stem}_{i}_{cell_name}.png"
                (out / r["split"]).mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(out / r["split"] / name), crop)
                rows.append({"file": f"{r['split']}/{name}", "split": r["split"], "cell": cell_name, "text": texts[cell_name]})
    df = pd.DataFrame(rows)
    df.to_csv(out / "labels.csv", index=False)
    return df

cells = crop_cells() if (DET / "index.csv").exists() else pd.DataFrame()
len(cells)